In [1]:
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'langchain',
    'langchain-community',
    'langchain-groq',
    'langchain-text-splitters',
    'langchain-huggingface',
    'faiss-cpu',
    'sentence-transformers',
    'pypdf',
    'spacy',
    'geopy',
    'folium',
    'streamlit',
    'streamlit-folium',
    'python-dotenv',
    'wikipedia',
    '-q'
])
print('✅ All packages installed!')

✅ All packages installed!


## Step 2: Verify Data Files

In [2]:
import os
from pathlib import Path

# ✅ FIXED PATH — matches your actual folder structure
DATA_DIR = Path(r'D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\Data')
VECTORSTORE_PATH = DATA_DIR.parent / 'vectorstore'

print('📂 Data directory:', DATA_DIR)
print('📂 Exists:', DATA_DIR.exists())
print()

if not DATA_DIR.exists():
    print('❌ ERROR: Data folder not found! Check the path above.')
else:
    pdfs = list(DATA_DIR.glob('*.pdf'))
    print(f'📄 Found {len(pdfs)} PDF files:')
    for pdf in pdfs:
        size_kb = pdf.stat().st_size // 1024
        print(f'   ✅ {pdf.name} ({size_kb} KB)')

📂 Data directory: D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\Data
📂 Exists: True

📄 Found 5 PDF files:
   ✅ fema_risk-map-phase-1-factsheet.pdf (373 KB)
   ✅ fema_risk-map-phase-2-factsheet.pdf (221 KB)
   ✅ fema_risk-map-phase-3-factsheet.pdf (220 KB)
   ✅ fema_risk-map-phase-4-factsheet.pdf (247 KB)
   ✅ fema_urban_flooding_guidance_for_homeowners_and_renters.pdf (3103 KB)


## Step 3: Load PDFs + Wikipedia → Build Vector Store
Run once. Reads all PDFs, adds Wikipedia data, chunks + embeds text, saves FAISS index locally.

In [3]:
from langchain_community.document_loaders import PyPDFLoader, WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ✅ FIXED PATH
DATA_DIR = Path(r'D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\Data')
VECTORSTORE_PATH = DATA_DIR.parent / 'vectorstore'

# ── 1. Load all PDFs ──────────────────────────────────────────────
docs = []
pdf_files = list(DATA_DIR.glob('*.pdf'))

if not pdf_files:
    raise FileNotFoundError(f'No PDF files found in {DATA_DIR}. Check the path in Step 2.')

for pdf_path in pdf_files:
    print(f'Loading PDF: {pdf_path.name} ...')
    loader = PyPDFLoader(str(pdf_path))
    loaded = loader.load()
    docs.extend(loaded)
    print(f'   → {len(loaded)} pages loaded')

print(f'\n✅ Total PDF pages: {len(docs)}')

# ── 2. Load Wikipedia city/flood data ────────────────────────────
wiki_topics = [
    'FEMA flood insurance program United States',
    'Flood zone mapping United States',
    'Chicago neighborhoods geography',
    'New York City flood risk',
    'Los Angeles urban planning',
    'Houston flooding geography',
    'National Flood Insurance Program',
    'Urban flooding causes prevention'
]

print('\nLoading Wikipedia articles...')
for topic in wiki_topics:
    try:
        loader = WikipediaLoader(query=topic, load_max_docs=1)
        wiki_docs = loader.load()
        docs.extend(wiki_docs)
        print(f'   ✅ {topic}')
    except Exception as e:
        print(f'   ⚠️  Skipped: {topic} ({e})')

print(f'\n✅ Total documents (PDF + Wikipedia): {len(docs)}')

# ── 3. Split into chunks ──────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', ' ', '']
)
chunks = splitter.split_documents(docs)
print(f'\n✅ Total chunks after splitting: {len(chunks)}')

# ── 4. Embed + Save to FAISS ──────────────────────────────────────
print('\nEmbedding chunks (this takes 1–2 minutes first time)...')
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)
vectorstore = FAISS.from_documents(chunks, embeddings)

VECTORSTORE_PATH.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(VECTORSTORE_PATH))

print(f'\n✅ Vector store saved to: {VECTORSTORE_PATH}')
print('🎉 Ingestion complete! You can now run the chatbot.')

C:\Users\Miraan Gee\AppData\Local\Temp\ipykernel_20212\319810115.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, WikipediaLoader
c:\Users\Miraan Gee\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Loading PDF: fema_risk-map-phase-1-factsheet.pdf ...
   → 2 pages loaded
Loading PDF: fema_risk-map-phase-2-factsheet.pdf ...
   → 2 pages loaded
Loading PDF: fema_risk-map-phase-3-factsheet.pdf ...
   → 2 pages loaded
Loading PDF: fema_risk-map-phase-4-factsheet.pdf ...
   → 2 pages loaded
Loading PDF: fema_urban_flooding_guidance_for_homeowners_and_renters.pdf ...
   → 51 pages loaded

✅ Total PDF pages: 59

Loading Wikipedia articles...
   ✅ FEMA flood insurance program United States
   ✅ Flood zone mapping United States
   ✅ Chicago neighborhoods geography
   ✅ New York City flood risk
   ✅ Los Angeles urban planning
   ✅ Houston flooding geography
   ✅ National Flood Insurance Program
   ✅ Urban flooding causes prevention

✅ Total documents (PDF + Wikipedia): 67

✅ Total chunks after splitting: 363

Embedding chunks (this takes 1–2 minutes first time)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


✅ Vector store saved to: D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\vectorstore
🎉 Ingestion complete! You can now run the chatbot.


## Step 4: Test the RAG Pipeline (without Streamlit)

> ⚠️ **Important:** Run Step 3 at least once before this cell so the vectorstore exists on disk.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path
from dotenv import load_dotenv
import os

# ✅ FIXED PATH — .env lives in Task4 folder
env_path = Path(r'D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\.env')

if not env_path.exists():
    raise FileNotFoundError(f'.env file not found at {env_path}\nCreate it with: GROQ_API_KEY=your_key_here')

load_dotenv(dotenv_path=env_path)

if not os.getenv('GROQ_API_KEY'):
    raise ValueError('GROQ_API_KEY not found in .env file!')

print('✅ API key loaded.')

# ✅ FIXED PATH
VECTORSTORE_PATH = Path(r'D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\vectorstore')

if not VECTORSTORE_PATH.exists():
    raise FileNotFoundError('Vectorstore not found. Please run Step 3 first!')

# ── Load vectorstore ──────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = FAISS.load_local(
    str(VECTORSTORE_PATH),
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# ✅ model= (not model_name=) — correct for current langchain-groq
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.2
)

# ── Build LCEL chain ──────────────────────────────────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = PromptTemplate.from_template("""
You are a helpful assistant that answers questions about flood zones, FEMA programs, and city geography.
Use the following context to answer the question. If you don't know, say so.

Context:
{context}

Question: {question}

Answer:""")

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('✅ RAG pipeline ready!\n')

# ── Test questions ────────────────────────────────────────────────
test_questions = [
    'What is the FEMA Risk MAP program?',
    'What happens during the flood map discovery phase?',
    'What are the flood risk zones in urban areas?',
    'How does FEMA notify communities about flood maps?'
]

for q in test_questions:
    print(f'❓ Question: {q}')
    answer = chain.invoke(q)
    print(f'💬 Answer: {answer}')
    print('-' * 80)

✅ API key loaded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ RAG pipeline ready!

❓ Question: What is the FEMA Risk MAP program?
💬 Answer: The FEMA Risk MAP program is the process used by the Federal Emergency Management Agency (FEMA) to create Flood Insurance Rate Maps, which identify areas at risk of flooding. The program involves a 4-phase approach: 

1. Discovery (Phase 1): Identifying risk
2. Risk Analysis and Mapping (Phase 2): Analyzing and mapping flood risks
3. Preliminary FIRM Release (Phase 3): Releasing preliminary flood maps to the public
4. (Phase 4 is not explicitly mentioned in the context, but it is implied to be part of the overall process)

The Risk MAP program aims to provide communities with detailed feedback on flood risks, inform planning and development, and offer resources and tools to help reduce flood risks.
--------------------------------------------------------------------------------
❓ Question: What happens during the flood map discovery phase?
💬 Answer: During the flood map discovery phase, the Risk MAP project

## Step 5: GIS Integration — Location Extraction + Map

First install the spaCy language model (run once):

In [5]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'
])
print('✅ spaCy model installed!')

✅ spaCy model installed!


In [6]:
import spacy
from geopy.geocoders import Nominatim
import folium
from IPython.display import display

nlp = spacy.load('en_core_web_sm')
geolocator = Nominatim(user_agent='geo_rag_chatbot_task4')

def extract_locations(text):
    doc = nlp(text)
    return list({ent.text for ent in doc.ents if ent.label_ in ['GPE', 'LOC', 'FAC']})

def geocode_locations(names):
    results = []
    for name in names:
        try:
            loc = geolocator.geocode(name, timeout=5)
            if loc:
                results.append({'name': name, 'lat': loc.latitude, 'lon': loc.longitude})
                print(f'   ✅ {name} → ({loc.latitude:.3f}, {loc.longitude:.3f})')
            else:
                print(f'   ⚠️  No result for: {name}')
        except Exception as e:
            print(f'   ⚠️  Could not geocode: {name} ({e})')
    return results

def build_map(locations, zoom=4):
    if not locations:
        return None
    center = [locations[0]['lat'], locations[0]['lon']]
    m = folium.Map(location=center, zoom_start=zoom, tiles='CartoDB positron')
    for loc in locations:
        folium.Marker(
            location=[loc['lat'], loc['lon']],
            popup=folium.Popup(loc['name'], max_width=200),
            tooltip=loc['name'],
            icon=folium.Icon(color='blue', icon='info-sign')
        ).add_to(m)
    return m

# Test with a sample answer
sample_text = """
FEMA's Risk MAP program works with communities across the United States including 
New York, Houston, Chicago, Los Angeles, and Miami to identify flood risk zones 
and update Flood Insurance Rate Maps.
"""

print('📍 Extracting locations...')
locs = extract_locations(sample_text)
print(f'Found: {locs}\n')

print('🌐 Geocoding...')
coords = geocode_locations(locs)

print('\n🗺️ Building map...')
m = build_map(coords, zoom=4)
if m:
    # ✅ FIXED PATH — save map inside Task4 folder
    map_path = r'D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\test_map.html'
    m.save(map_path)
    print(f'✅ Map saved to: {map_path}')
    display(m)
else:
    print('⚠️ No locations to map')

📍 Extracting locations...
Found: ['the United States', 'Los Angeles', 'Chicago', 'New York', 'Miami', 'Houston']

🌐 Geocoding...
   ✅ the United States → (36.134, -80.278)
   ✅ Los Angeles → (34.054, -118.243)
   ✅ Chicago → (41.876, -87.624)
   ✅ New York → (40.713, -74.006)
   ✅ Miami → (25.774, -80.194)
   ✅ Houston → (29.759, -95.368)

🗺️ Building map...
✅ Map saved to: D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\test_map.html


## Step 6: Launch Streamlit App

The full interactive chatbot runs as a Streamlit app.

**Open a terminal / Anaconda Prompt and run:**

```bash
cd "D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4"
streamlit run app.py
```

This will open your browser at `http://localhost:8501`

In [7]:
# Optional: launch directly from notebook
import subprocess
from pathlib import Path

# ✅ FIXED PATH
app_path = Path(r'D:\OneDrive - National University of Sciences & Technology\internship\geoai\Task4\app.py')

if not app_path.exists():
    print(f'❌ app.py not found at {app_path}')
    print('   Create app.py in your Task4 folder first, then re-run this cell.')
else:
    subprocess.Popen(['streamlit', 'run', str(app_path)])
    print('✅ Streamlit app launching... open http://localhost:8501')

✅ Streamlit app launching... open http://localhost:8501
